<a href="https://colab.research.google.com/github/muhammetalicvs-prog/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Data contract

1. **What does one row mean?**  
   One row in the final feature frame represents one pseudonymized content page at the end of the March 2026 decision window.

2. **Which table will I use?**  
   I will use the `fact_content_daily_performance` table. Its source grain is one row per report date, pseudonymized client, and pseudonymized content page.

3. **Which time window will I use?**  
   March 2026 is the feature window. April 2026 is used only as the forward outcome window. June 2026 remains sealed as the final test month.

4. **What will I predict or rank?**  
   I will rank content pages for human refresh review using a forward decline proxy. The proxy is 1 when April 2026 search impressions are lower than March 2026 search impressions, and 0 otherwise.

5. **What will I deliberately exclude?**  
   I will exclude April outcome values, label-derived columns, URLs, client names, private queries, and pseudonymous identifiers from the model features. These fields are either unavailable at the decision moment, unsafe to publish, or unsuitable as transferable predictive signals.

In [33]:
# Install the packages required for querying the Hugging Face warehouse.
# - DuckDB runs SQL directly over remote Parquet files.
# - huggingface_hub supports authenticated Hugging Face access.

!pip -q install --upgrade duckdb huggingface_hub

import os
import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata
from IPython.display import display

# Read the Hugging Face token from Colab Secrets.
# The token is never printed or written into the notebook.
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. Add it from the Colab key panel "
        "before running this notebook."
    )

# Make the token available to Hugging Face-compatible libraries.
os.environ["HF_TOKEN"] = HF_TOKEN

# Start an in-memory DuckDB connection.
con = duckdb.connect(database=":memory:")

# Install and load remote file support.
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

# Safely escape any single quote that might occur in the token.
safe_token = HF_TOKEN.replace("'", "''")

# Create a temporary Hugging Face secret inside DuckDB.
# The token value is not displayed in notebook output.
con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
""")

# Base warehouse location.
WAREHOUSE_ROOT = "hf://datasets/FlyRank/internship-warehouse"

# We use a mid-panel month for features and the following month for the label.
FEATURE_MONTH = "2026-03"
OUTCOME_MONTH = "2026-04"

# Read only the required monthly partitions, not the entire 79M-row table.
FEATURE_PATH = (
    f"{WAREHOUSE_ROOT}/fact_content_daily_performance/"
    f"month={FEATURE_MONTH}/*.parquet"
)

OUTCOME_PATH = (
    f"{WAREHOUSE_ROOT}/fact_content_daily_performance/"
    f"month={OUTCOME_MONTH}/*.parquet"
)

print("DuckDB connection created.")
print("Feature month:", FEATURE_MONTH)
print("Outcome month:", OUTCOME_MONTH)
print("The Hugging Face token was loaded securely and was not printed.")

DuckDB connection created.
Feature month: 2026-03
Outcome month: 2026-04
The Hugging Face token was loaded securely and was not printed.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field classification

#### Features

The final feature frame contains exactly five features:

1. `log_impressions` — logarithm of total March search impressions.
2. `log_clicks` — logarithm of total March search clicks.
3. `weighted_ctr` — total March clicks divided by total March impressions.
4. `weighted_avg_position` — impression-weighted average search position during March.
5. `active_search_days` — number of March dates on which the page recorded at least one search impression.

#### Label / proxy

- `decline_proxy` — equals 1 when April search impressions are lower than March search impressions, and 0 otherwise.
- This is a forward operational proxy for prioritization, not proof that the content itself caused the decline.

#### Context

- `client_id` — used for grouping, joins, and future client-grouped validation.
- `content_id` — used to connect daily observations belonging to the same content page.
- `feature_month` — identifies the completed observation window.
- `outcome_month` — identifies the forward label window.
- `march_impressions` and `april_impressions` — retained temporarily to construct and audit the proxy, but not used as model inputs.

#### Excluded

- April outcome measurements are excluded from honest features because they are not knowable at the March decision moment.
- `decline_proxy` and columns derived from it are excluded because they reveal the answer.
- `client_id` and `content_id` are excluded from the feature list because they are identifiers rather than transferable page signals.
- Client names, domains, URLs, raw queries, and private text are excluded for privacy and because they are not needed for this lane.
- June 2026 is excluded from development because it is the final month and should remain sealed for later testing.

In [34]:
# Inspect the warehouse schema before writing the contract queries.
# This prevents us from guessing field names.

schema_df = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{FEATURE_PATH}', hive_partitioning = TRUE)
""").df()

display(schema_df)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [35]:
required_columns = {
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_data_available",
}

available_columns = set(schema_df["column_name"].tolist())
missing_columns = required_columns - available_columns

if missing_columns:
    raise ValueError(
        f"Required warehouse columns are missing: {sorted(missing_columns)}"
    )

print("All required columns are available.")

All required columns are available.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification Query 1 — Source grain

The declared source grain is one row per `report_date × client_id × content_id`. If the query below returns zero rows, no duplicate groups were found in the March 2026 slice.

In [36]:
# Verification Query 1:
# Check whether any report_date × client_hash_id × content_hash_id group
# appears more than once in the March feature partition.

grain_check = con.execute(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS rows_in_group
    FROM read_parquet(
        '{FEATURE_PATH}',
        hive_partitioning = TRUE
    )
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    ORDER BY rows_in_group DESC
    LIMIT 10
""").df()

display(grain_check)

if grain_check.empty:
    print(
        "Result: zero duplicate groups were found. "
        "The declared daily source grain holds for this slice."
    )
else:
    print(
        "Warning: duplicate grain groups were found. "
        "The contract must be reviewed."
    )


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,rows_in_group


Result: zero duplicate groups were found. The declared daily source grain holds for this slice.


### Verification Query 2 — Slice size and date span

This query measures the number of observed daily rows, pseudonymized clients, pseudonymized content pages, and the actual date range in the March 2026 feature slice.

In [37]:
# Verification Query 2:
# Measure the size and observed date range of the March slice.

slice_summary = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT client_hash_id) AS client_count,
        COUNT(DISTINCT content_hash_id) AS content_count,
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date
    FROM read_parquet(
        '{FEATURE_PATH}',
        hive_partitioning = TRUE
    )
""").df()

display(slice_summary)

summary_row = slice_summary.iloc[0]

print(
    f"The March 2026 slice contains "
    f"{int(summary_row['row_count']):,} observed daily rows, "
    f"{int(summary_row['content_count']):,} pseudonymized content pages, "
    f"and {int(summary_row['client_count']):,} pseudonymized clients."
)

print(
    f"The observed date span is "
    f"{summary_row['min_report_date']} to "
    f"{summary_row['max_report_date']}."
)

,row_count,client_count,content_count,min_report_date,max_report_date
0,9841378,55,331437,2026-03-01,2026-03-31


The March 2026 slice contains 9,841,378 observed daily rows, 331,437 pseudonymized content pages, and 55 pseudonymized clients.
The observed date span is 2026-03-01 00:00:00 to 2026-03-31 00:00:00.


### Verification Query 3 — GA4 availability

GA4 values can be zero-filled before analytics tracking becomes available. Therefore, zero does not always mean measured zero engagement. The query below uses `ga4_data_available IS TRUE` and counts how many March rows survive the availability filter.

In [38]:
# Verification Query 3:
# Count how many rows have genuinely available GA4 measurements.
# The assignment explicitly requires the use of IS TRUE.

availability_summary = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS available_rows,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS NOT TRUE
        ) AS unavailable_rows,

        ROUND(
            100.0
            * COUNT(*) FILTER (
                WHERE ga4_data_available IS TRUE
            )
            / NULLIF(COUNT(*), 0),
            2
        ) AS available_percentage

    FROM read_parquet(
        '{FEATURE_PATH}',
        hive_partitioning = TRUE
    )
""").df()

display(availability_summary)

availability_row = availability_summary.iloc[0]

print(
    f"After applying ga4_data_available IS TRUE, "
    f"{int(availability_row['available_rows']):,} of "
    f"{int(availability_row['total_rows']):,} rows remain "
    f"({availability_row['available_percentage']:.2f}%)."
)

print(
    "Rows without available GA4 data are not interpreted "
    "as measured zero engagement."
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,available_rows,unavailable_rows,available_percentage
0,9841378,413966,9427412,4.21


After applying ga4_data_available IS TRUE, 413,966 of 9,841,378 rows remain (4.21%).
Rows without available GA4 data are not interpreted as measured zero engagement.


### Five-feature frame

The feature window is March 2026. Every feature is calculated only from observations available by the end of March. April is joined afterward only to construct the forward decline proxy.

#### Feature availability

1. **`log_impressions`**  
   Knowable at the decision moment because it is calculated only from search impressions observed during the completed March feature window.

2. **`log_clicks`**  
   Knowable at the decision moment because it is calculated only from search clicks observed during the completed March feature window.

3. **`weighted_ctr`**  
   Knowable at the decision moment because both its clicks and impressions were measured during March.

4. **`weighted_avg_position`**  
   Knowable at the decision moment because it summarizes search positions already observed during March.

5. **`active_search_days`**  
   Knowable at the decision moment because it counts the March dates on which search visibility was observed.

The April outcome fields are used only to construct and audit the forward proxy. They are not honest features.

In [39]:
# Build one page-level row for the March decision point.
#
# March:
#   Used to calculate the five honest features.
#
# April:
#   Used only to create the forward decline proxy.

feature_frame = con.execute(f"""
    WITH march_page AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(COALESCE(gsc_impressions, 0)) AS march_impressions,
            SUM(COALESCE(gsc_clicks, 0)) AS march_clicks,

            CASE
                WHEN SUM(COALESCE(gsc_impressions, 0)) > 0
                THEN
                    SUM(COALESCE(gsc_clicks, 0)) * 1.0
                    / SUM(COALESCE(gsc_impressions, 0))
                ELSE NULL
            END AS weighted_ctr,

            CASE
                WHEN SUM(COALESCE(gsc_impressions, 0)) > 0
                THEN
                    SUM(
                        COALESCE(gsc_avg_position, 0)
                        * COALESCE(gsc_impressions, 0)
                    ) * 1.0
                    / SUM(COALESCE(gsc_impressions, 0))
                ELSE NULL
            END AS weighted_avg_position,

            COUNT(
                DISTINCT CASE
                    WHEN COALESCE(gsc_impressions, 0) > 0
                    THEN report_date
                END
            ) AS active_search_days

        FROM read_parquet(
            '{FEATURE_PATH}',
            hive_partitioning = TRUE
        )

        GROUP BY
            client_hash_id,
            content_hash_id

        HAVING SUM(COALESCE(gsc_impressions, 0)) > 0
    ),

    april_page AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(COALESCE(gsc_impressions, 0)) AS april_impressions

        FROM read_parquet(
            '{OUTCOME_PATH}',
            hive_partitioning = TRUE
        )

        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        m.client_hash_id,
        m.content_hash_id,

        LN(1 + m.march_impressions) AS log_impressions,
        LN(1 + m.march_clicks) AS log_clicks,
        m.weighted_ctr,
        m.weighted_avg_position,
        m.active_search_days,

        m.march_impressions,
        a.april_impressions,

        CASE
            WHEN a.april_impressions < m.march_impressions
            THEN 1
            ELSE 0
        END AS decline_proxy,

        '{FEATURE_MONTH}' AS feature_month,
        '{OUTCOME_MONTH}' AS outcome_month

    FROM march_page AS m

    INNER JOIN april_page AS a
        ON m.client_hash_id = a.client_hash_id
        AND m.content_hash_id = a.content_hash_id
""").df()

print("Feature frame shape:", feature_frame.shape)
display(feature_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (176737, 12)


,client_hash_id,content_hash_id,log_impressions,log_clicks,weighted_ctr,weighted_avg_position,active_search_days,march_impressions,april_impressions,decline_proxy,feature_month,outcome_month
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,5.846439,0.693147,0.002899,23.492754,31,345.0,187.0,1,2026-03,2026-04
1,client_62f4a7e64f5e0096,content_13a8105125458098,3.637586,0.693147,0.027027,3.756757,17,37.0,23.0,1,2026-03,2026-04
2,client_62f4a7e64f5e0096,content_6a887d56ab6c8362,2.197225,0.000000,0.000000,12.000000,5,8.0,8.0,0,2026-03,2026-04
3,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,2.772589,0.000000,0.000000,5.333333,8,15.0,17.0,0,2026-03,2026-04
4,client_62f4a7e64f5e0096,content_86ab16840c4e0e1a,6.751101,0.693147,0.001171,8.152225,31,854.0,202.0,1,2026-03,2026-04


In [40]:
honest_features = [
    "log_impressions",
    "log_clicks",
    "weighted_ctr",
    "weighted_avg_position",
    "active_search_days",
]

target_column = "decline_proxy"

assert len(honest_features) == 5, "The assignment requires exactly five features."

missing_feature_columns = [
    column
    for column in honest_features
    if column not in feature_frame.columns
]

if missing_feature_columns:
    raise ValueError(
        f"Missing feature columns: {missing_feature_columns}"
    )

print("Exactly five honest features are defined:")
for number, feature in enumerate(honest_features, start=1):
    print(f"{number}. {feature}")

Exactly five honest features are defined:
1. log_impressions
2. log_clicks
3. weighted_ctr
4. weighted_avg_position
5. active_search_days


In [41]:
# Check class balance and missing values before the quick experiment.

quality_summary = pd.DataFrame({
    "column": honest_features + [target_column],
    "missing_count": [
        int(feature_frame[column].isna().sum())
        for column in honest_features + [target_column]
    ],
    "missing_percentage": [
        round(
            100 * feature_frame[column].isna().mean(),
            2
        )
        for column in honest_features + [target_column]
    ],
})

display(quality_summary)

print("Decline proxy distribution:")
display(
    feature_frame[target_column]
    .value_counts(dropna=False)
    .rename_axis("decline_proxy")
    .reset_index(name="row_count")
)

print(
    "Positive-label rate:",
    round(feature_frame[target_column].mean(), 4)
)

,column,missing_count,missing_percentage
0,log_impressions,0,0.0
1,log_clicks,0,0.0
2,weighted_ctr,0,0.0
3,weighted_avg_position,0,0.0
4,active_search_days,0,0.0
5,decline_proxy,0,0.0


Decline proxy distribution:


,decline_proxy,row_count
0,1,111967
1,0,64770


Positive-label rate: 0.6335


### Deliberate leakage experiment

First, I calculate a quick honest score using only the five March features. Then I deliberately add one label-derived column named `leaked_label_copy`. This column directly copies the decline proxy and therefore reveals the answer.

The leaky score is expected to move toward a perfect result. That improvement is not real predictive performance. It is evidence that the evaluation has been contaminated by information derived from the label.

After demonstrating the problem, I delete the leaked column and retain only the honest feature set.

In [42]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Work on a copy so that the main feature frame can be protected.
experiment_df = feature_frame.copy()

# Keep only rows with a valid binary target.
experiment_df = experiment_df[
    experiment_df[target_column].isin([0, 1])
].copy()

# The quick experiment requires both classes.
if experiment_df[target_column].nunique() < 2:
    raise ValueError(
        "The decline proxy contains only one class. "
        "AUC cannot be calculated."
    )

def calculate_quick_auc(dataframe, feature_columns, target):
    """
    Fit and score a small model on the same frame.

    This is intentionally only a quick leakage demonstration.
    It is not an honest estimate of future model performance.
    Later modeling work must use a proper time-based or
    client-grouped validation strategy.
    """

    X = dataframe[feature_columns]
    y = dataframe[target]

    model = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter=2000,
                    random_state=42
                )
            ),
        ]
    )

    model.fit(X, y)
    predicted_probability = model.predict_proba(X)[:, 1]

    return roc_auc_score(y, predicted_probability)

# Honest score: only the five March features.
honest_auc = calculate_quick_auc(
    dataframe=experiment_df,
    feature_columns=honest_features,
    target=target_column,
)

print(f"Honest quick AUC: {honest_auc:.4f}")

Honest quick AUC: 0.6102


In [43]:
# DELIBERATE LEAK:
# This column directly copies the answer.
# It must never remain in the final feature frame.

experiment_df["leaked_label_copy"] = experiment_df[target_column]

leaky_features = honest_features + ["leaked_label_copy"]

leaky_auc = calculate_quick_auc(
    dataframe=experiment_df,
    feature_columns=leaky_features,
    target=target_column,
)

print(f"Honest quick AUC: {honest_auc:.4f}")
print(f"Leaky quick AUC:  {leaky_auc:.4f}")
print(f"Artificial jump:  {leaky_auc - honest_auc:.4f}")

Honest quick AUC: 0.6102
Leaky quick AUC:  1.0000
Artificial jump:  0.3898


In [44]:
# Remove the deliberately leaked field immediately.

experiment_df = experiment_df.drop(
    columns=["leaked_label_copy"]
)

assert "leaked_label_copy" not in experiment_df.columns
assert set(honest_features).issubset(experiment_df.columns)

print("Leakage column removed successfully.")
print(f"Retained honest quick AUC: {honest_auc:.4f}")
print("Final feature count:", len(honest_features))
print("Final honest features:", honest_features)

Leakage column removed successfully.
Retained honest quick AUC: 0.6102
Final feature count: 5
Final honest features: ['log_impressions', 'log_clicks', 'weighted_ctr', 'weighted_avg_position', 'active_search_days']


### Leakage result

The quick score increased toward a perfect result after `leaked_label_copy` was added. This happened because the column directly revealed the decline proxy rather than providing information available at the March decision moment.

The leaky result is invalid and is not retained as model performance. I deleted the label-derived column and kept the honest five-feature result.

The honest quick AUC is also not a final performance claim because this small demonstration fits and scores on the same feature frame. It is used only to make the leakage effect visible. A later modeling notebook should use honest time-based evaluation and client-aware splitting.

In [45]:
# Create the final safe modeling frame.
#
# IDs remain only as context for grouping and future splitting.
# The model input itself contains exactly the five honest features.

final_feature_frame = feature_frame[
    [
        "client_hash_id",
        "content_hash_id",
        "feature_month",
        "outcome_month",
        *honest_features,
        target_column,
    ]
].copy()

assert len(honest_features) == 5
assert "march_impressions" not in final_feature_frame.columns
assert "april_impressions" not in final_feature_frame.columns
assert "leaked_label_copy" not in final_feature_frame.columns

print("Final frame shape:", final_feature_frame.shape)
print("Final modeling features:", honest_features)

display(final_feature_frame.head())

Final frame shape: (176737, 10)
Final modeling features: ['log_impressions', 'log_clicks', 'weighted_ctr', 'weighted_avg_position', 'active_search_days']


,client_hash_id,content_hash_id,feature_month,outcome_month,log_impressions,log_clicks,weighted_ctr,weighted_avg_position,active_search_days,decline_proxy
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,2026-03,2026-04,5.846439,0.693147,0.002899,23.492754,31,1
1,client_62f4a7e64f5e0096,content_13a8105125458098,2026-03,2026-04,3.637586,0.693147,0.027027,3.756757,17,1
2,client_62f4a7e64f5e0096,content_6a887d56ab6c8362,2026-03,2026-04,2.197225,0.000000,0.000000,12.000000,5,0
3,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,2026-03,2026-04,2.772589,0.000000,0.000000,5.333333,8,0
4,client_62f4a7e64f5e0096,content_86ab16840c4e0e1a,2026-03,2026-04,6.751101,0.693147,0.001171,8.152225,31,1


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Named limitation

A key limitation of this slice is that the warehouse is an unbalanced panel. Clients do not all have the same amount of historical Search Console and Analytics coverage. Therefore, the March-to-April frame may represent clients and pages with longer measurement histories more strongly than recently connected clients.

Rows before a client's GA4 start date may contain zero-filled analytics values while `ga4_data_available` is false. Those zeros cannot be interpreted as measured zero engagement.

The proxy also measures an observed month-to-month change in search impressions. It cannot explain why the change occurred. A decline may reflect seasonality, changing demand, ranking changes, tracking differences, or other external factors. The result is directional decision-support for human review, not a causal conclusion.

In [46]:
# Final automated checks for the notebook contract.

checks = {
    "feature_frame_has_rows": len(final_feature_frame) > 0,

    "exactly_five_features": len(honest_features) == 5,

    "target_exists": (
        target_column in final_feature_frame.columns
    ),

    "target_is_binary": set(
        final_feature_frame[target_column]
        .dropna()
        .unique()
    ).issubset({0, 1}),

    "leak_column_removed": (
        "leaked_label_copy"
        not in final_feature_frame.columns
    ),

    "outcome_impressions_excluded": (
        "april_impressions"
        not in final_feature_frame.columns
    ),

    "raw_march_impressions_excluded": (
        "march_impressions"
        not in final_feature_frame.columns
    ),

    "ids_not_in_feature_list": (
        "client_id" not in honest_features
        and "content_id" not in honest_features
    ),

    "sealed_june_not_used": (
        FEATURE_MONTH != "2026-06"
        and OUTCOME_MONTH != "2026-06"
    ),
}

check_df = pd.DataFrame(
    {
        "check": list(checks.keys()),
        "passed": list(checks.values()),
    }
)

display(check_df)

if all(checks.values()):
    print("All automated contract checks passed.")
else:
    failed_checks = [
        name
        for name, passed in checks.items()
        if not passed
    ]

    raise AssertionError(
        f"Failed contract checks: {failed_checks}"
    )

,check,passed
0,feature_frame_has_rows,True
1,exactly_five_features,True
2,target_exists,True
3,target_is_binary,True
4,leak_column_removed,True
5,outcome_impressions_excluded,True
6,raw_march_impressions_excluded,True
7,ids_not_in_feature_list,True
8,sealed_june_not_used,True


All automated contract checks passed.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.